# Gold　Layer

## 目的・方針

- `workspace.silver` の3テーブル（products / sales / inventory）を結合し、ビジネス指標を算出してGold層として提供する
- 作成するテーブル
    - `_30_gold_daily_sales_summary`: 日次売上サマリ（sale_date × store × product/category 単位の売上集計）
    - `_30_gold_inventory_status`: 在庫ステータス（store × product 単位の在庫評価額・欠品/低在庫フラグ）
    - `_30_gold_product_ranking`: 商品別販売実績ランキング（product_id 単位の売上集計とランキング）
- テーブル間の結合はGold層の責務とする（Silver層の設計方針を継承）
- ソース（Silver）はフルスナップショット想定のため、書き込みは `overwrite` とする（再実行しても結果が変わらない = 冪等）
- 結合前に Silver のリネージュ列（`_ingested_at`, `_silver_processed_at`）は drop する（結合すると同名列が重複し、参照が曖昧になるため）。Gold固有の処理時刻として `_gold_processed_at` を新規付与する

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, sum as _sum, current_timestamp, dense_rank, desc

SILVER_SCHEMA = "workspace.silver"
GOLD_SCHEMA = "workspace.gold"
GOLD_TABLES = ["daily_sales_summary", "inventory_status", "product_ranking"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

LOW_STOCK_THRESHOLD = 10  # 店舗×商品の在庫がこの値未満なら低在庫とみなす（暫定値）


def read_silver(table_name: str):
    """Silverテーブルを読み込み、リネージュ列（結合時に衝突するため）を落とす"""
    df = spark.read.table(f"{SILVER_SCHEMA}._20_silver_{table_name}")
    return df.drop("_ingested_at", "_silver_processed_at")

In [0]:
def build_daily_sales_summary(df_sales, df_products):
    """
    日次売上サマリ
    - sales × products を product_id で結合
    - sale_date × store × product（category含む）単位で数量・売上金額を集計
    """
    df = (
        df_sales.join(df_products, on="product_id", how="left")
        .groupBy(
            "sale_date", "store_id", "store_name",
            "product_id", "product_name", "category",
        )
        .agg(
            _sum("quantity").alias("total_quantity"),
            _sum("sales_amount").alias("total_sales_amount"),
        )
    )
    return df

In [0]:
def build_inventory_status(df_inventory, df_products):
    """
    在庫ステータス
    - inventory × products を product_id で結合
    - 在庫評価額（stock_quantity × unit_price）を算出
    - out_of_stock: 在庫0、low_stock: 0 < 在庫 < LOW_STOCK_THRESHOLD
    """
    df = (
        df_inventory.join(df_products, on="product_id", how="left")
        .withColumn("stock_value", col("stock_quantity") * col("unit_price"))
        .withColumn("out_of_stock", col("stock_quantity") == 0)
        .withColumn(
            "low_stock",
            (col("stock_quantity") > 0) & (col("stock_quantity") < LOW_STOCK_THRESHOLD),
        )
        .select(
            "store_id", "store_name",
            "product_id", "product_name", "category",
            "stock_quantity", "unit_price", "stock_value",
            "updated_at", "out_of_stock", "low_stock",
        )
    )
    return df

In [0]:
def build_product_ranking(df_sales, df_products):
    """
    商品別販売実績ランキング
    - sales を product_id 単位で集計（全店舗・全期間合計）
    - products と結合して商品名・カテゴリを付与
    - 売上金額（total_revenue）降順で dense_rank を付与
    """
    df_agg = df_sales.groupBy("product_id").agg(
        _sum("quantity").alias("total_quantity"),
        _sum("sales_amount").alias("total_revenue"),
    )

    window_spec = Window.orderBy(desc("total_revenue"))

    df = (
        df_agg.join(df_products, on="product_id", how="left")
        .withColumn("revenue_rank", dense_rank().over(window_spec))
        .select(
            "revenue_rank", "product_id", "product_name", "category",
            "total_quantity", "total_revenue",
        )
    )
    return df

In [0]:
GOLD_BUILDERS = {
    "daily_sales_summary": lambda: build_daily_sales_summary(
        read_silver("sales"), read_silver("products")
    ),
    "inventory_status": lambda: build_inventory_status(
        read_silver("inventory"), read_silver("products")
    ),
    "product_ranking": lambda: build_product_ranking(
        read_silver("sales"), read_silver("products")
    ),
}


def write_to_gold(table_name: str, df) -> None:
    """
    Goldテーブルへの書き込み共通処理

    goldテーブル _30_ の接頭辞を付与
    """
    target_table = f"{GOLD_SCHEMA}._30_gold_{table_name}"

    df_gold = df.withColumn("_gold_processed_at", current_timestamp())
    row_count = df_gold.count()

    (
        df_gold.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Gold書き込み完了: {target_table}（{row_count}件）")


for table_name in GOLD_TABLES:
    write_to_gold(table_name, GOLD_BUILDERS[table_name]())

In [0]:
# 書き込み結果の確認
for table_name in GOLD_TABLES:
    df = spark.read.table(f"{GOLD_SCHEMA}._30_gold_{table_name}")
    print(f"{table_name}: {df.count()}件")
    df.printSchema()
    display(df)